# Full Report — Executive Summary

**Title:** AI Writing Forensics: Mediaite Investigation

**Forensic question:** What are the conclusions and how strong is the evidence?

**Input artifacts:** Per-author `data/analysis/<slug>_result.json` files emitted by chapters 05–08 (change-point detection, embedding drift, statistical evidence, control comparison).

**Output artifacts:** This chapter (executive narrative; tables computed live from the per-author result bundles).

**Run metadata:** Provenance hashes are emitted by the final code cell (git SHA, analysis-config hash, run timestamp). See chapter 00 for the canonical corpus hash and `data/analysis/corpus_custody.json` for the chain-of-custody record.

## Executive summary

The post-Phase-15 + Fix-A–G + manifest-regen pass restores meaningful signal across all three detection pipelines and surfaces a single defensible single-author finding. Four headline findings:

1. **Pipeline B (embedding drift) is alive again.** All 12 named authors now produce a non-zero `pb_max` after the embedding manifest was regenerated, recovering from the pre-fix state in which only 1/14 authors had any drift signal. Top-five `pb_max` (named authors only): `david-gilmour` (0.598), `colby-hall` (0.589), `sarah-rumpf` (0.587), `charlie-nash` (0.537), `joe-depaolo` (0.536). All five are individual journalists and warrant follow-up.

2. **FDR-significance is restored across all 12 named authors.** Under the new per-family Benjamini–Hochberg correction, **10,874 / 111,560 hypothesis tests are significant (9.7%)**. Pre-Phase-15 the same harness yielded **0/2,277** significant tests because the global BH correction was running over an under-populated test pool. Every one of the 12 named authors now contributes some significant evidence rather than the entire set collapsing to noise.

3. **Single-author headline: `colby-hall`.** This is the only author where Pipeline A (stylometric convergence) and Pipeline B (embedding drift) cross threshold together for ≥100 windows. PA_max = 0.94, PB_max = 0.59, with **170 AB-confirmed convergence windows** (`'ab'` in `passes_via`). The window of interest — `2026-01-08 → 2026-04-08` — has five families converging simultaneously: `ai_markers`, `entropy`, `lexical_richness`, `readability`, `self_similarity`.

4. **Performance is back inside a workable budget.** Full analysis across 12 named authors (serial) now runs in ~21 minutes versus multiple hours pre-F0, when the PELT RBF kernel was responsible for 99.2% of wall-clock time. This makes iteration on prereg thresholds, control comparisons, and family weighting tractable rather than batch-overnight.

**Caveats** (read before citing any number above):

- **Pipeline C (token-probability convergence) is still 0.0 across the board.** This was not in scope for this round; lifting it requires Phase 9 token-probability data that has not yet landed.
- Pre-registration is **not yet locked**. Until that lock exists, every threshold choice in this notebook is exploratory, not confirmatory.

## Top-line metrics: pre-fix vs post-fix

The next code cell loads every per-author `result.json` from `data/analysis/` and recomputes the headline metrics so the table below is sourced from the same artifacts the rest of the report consumes.

In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype+notebook_connected+png'


In [ ]:
import json

import polars as pl
from IPython.display import Markdown, display

from forensics.config import get_project_root

ANALYSIS_DIR = get_project_root() / "data" / "analysis"
SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
RESULT_FILES = [
    p
    for p in sorted(ANALYSIS_DIR.glob("*_result.json"))
    if p.name.removesuffix("_result.json") not in SHARED_BYLINE_SLUGS
]

rows: list[dict[str, object]] = []
for path in RESULT_FILES:
    slug = path.name.removesuffix("_result.json")
    payload = json.loads(path.read_text(encoding="utf-8"))
    tests = payload.get("hypothesis_tests", []) or []
    cw = payload.get("convergence_windows", []) or []
    pa_vals = [(w.get("pipeline_a_score") or 0.0) for w in cw]
    pb_vals = [(w.get("pipeline_b_score") or 0.0) for w in cw]
    ratio_vals = [(w.get("convergence_ratio") or 0.0) for w in cw]
    ab_windows = sum(1 for w in cw if "ab" in (w.get("passes_via") or []))
    rows.append(
        {
            "slug": slug,
            "n_tests": len(tests),
            "n_sig": sum(1 for t in tests if t.get("significant")),
            "pa_max": max(pa_vals) if pa_vals else 0.0,
            "pb_max": max(pb_vals) if pb_vals else 0.0,
            "ratio_max": max(ratio_vals) if ratio_vals else 0.0,
            "n_windows": len(cw),
            "n_ab_windows": ab_windows,
        }
    )

summary = pl.DataFrame(rows)

n_authors = summary.height
authors_pb_pos = int((summary["pb_max"] > 0).sum())
total_tests = int(summary["n_tests"].sum())
total_sig = int(summary["n_sig"].sum())
ratio_max_overall = float(summary["ratio_max"].max())

colby = summary.filter(pl.col("slug") == "colby-hall")
if colby.height == 1:
    colby_pa = float(colby["pa_max"][0])
    colby_pb = float(colby["pb_max"][0])
    colby_ab = int(colby["n_ab_windows"][0])
else:
    colby_pa = colby_pb = float("nan")
    colby_ab = 0

metrics_md = f"""| metric | pre-fix | post-fix |
|---|---|---|
| authors with PB_max > 0 | 1/14 | {authors_pb_pos}/{n_authors} |
| significant tests (BH-FDR) | 0/2,277 | {total_sig:,}/{total_tests:,} |
| colby-hall PB_max | 0.00 | {colby_pb:.3f} |
| colby-hall PA_max | 0.94 | {colby_pa:.2f} |
| convergence ratio max | 0.75 | {ratio_max_overall:.2f} |
| colby-hall AB-confirmed windows | n/a | {colby_ab} |
"""
display(Markdown(metrics_md))

display(
    summary.sort("pb_max", descending=True).select(
        ["slug", "pa_max", "pb_max", "ratio_max", "n_ab_windows", "n_sig", "n_tests"]
    )
)

## Finding strength classification

Each author's strongest convergence window is graded with `classify_finding_strength` from `forensics.models.report`. Hypothesis tests passed to the classifier are **window-scoped** — only tests whose `feature_name` appears in the window's `features_converging` list count toward the tally. This prevents an author from being credited for significant tests that don't actually support the convergence window in question.

A **strong significant test** is one with corrected p < 0.01 *and* |Cohen's d| ≥ 0.8.

- **STRONG (red):** ≥3 strong significant tests *and* controls clean (`editorial_vs_author_signal > 0.7`) *and* (Pipeline C ok when probability features are available).
- **MODERATE (orange):** ≥3 significant tests *and* ≥1 strong significant test. The `≥1 strong` floor distinguishes meaningful from marginal evidence; the `≥3 sig` count keeps the bar above the trivial 2-test threshold that previously bucketed authors with vastly different evidence strength into the same tier.
- **WEAK (yellow):** ≥1 significant test.
- **NONE (green):** no significant tests in this window's features.

Probability features are unavailable in this run, so the Pipeline C clause is a no-op. The `editorial_vs_author_signal` is looked up per-target from `data/analysis/comparison_report.json` (`targets.<slug>.editorial_vs_author_signal`); controls have no such signal and default to 0.0, which structurally reserves STRONG for designated targets.

**Threshold provenance caveat:** the MODERATE bar (`≥3 sig AND ≥1 strong`) was chosen after observing that the prior `≥2 sig` bar bucketed authors with up to a 50× range in evidence quality together. This is exploratory; the bar must be locked in `data/preregistration/` before any confirmatory claim is made.

In [ ]:
import json

import polars as pl
from IPython.display import display

from forensics.config import get_project_root
from forensics.models.analysis import ConvergenceWindow, HypothesisTest
from forensics.models.report import classify_finding_strength

ANALYSIS_DIR = get_project_root() / "data" / "analysis"
comparison_path = ANALYSIS_DIR / "comparison_report.json"
if comparison_path.is_file():
    comparison_blob = json.loads(comparison_path.read_text(encoding="utf-8"))
else:
    comparison_blob = {"targets": {}}

strength_rows: list[dict[str, object]] = []
SHARED_BYLINE_SLUGS = {"mediaite", "mediaite-staff"}
for path in sorted(ANALYSIS_DIR.glob("*_result.json")):
    if path.name.removesuffix("_result.json") in SHARED_BYLINE_SLUGS:
        continue
    slug = path.name.removesuffix("_result.json")
    payload = json.loads(path.read_text(encoding="utf-8"))
    raw_windows = payload.get("convergence_windows", []) or []
    if not raw_windows:
        strength_rows.append({"slug": slug, "strength": "none", "pa": 0.0, "pb": 0.0, "n_sig": 0})
        continue
    windows = [ConvergenceWindow.model_validate(w) for w in raw_windows]
    best = max(
        windows,
        key=lambda w: ((w.pipeline_a_score or 0.0) + (w.pipeline_b_score or 0.0)),
    )
    tests = [HypothesisTest.model_validate(t) for t in payload.get("hypothesis_tests", []) or []]
    window_features = set(best.features_converging or [])
    window_tests = (
        [t for t in tests if t.feature_name in window_features] if window_features else tests
    )
    target_block = comparison_blob.get("targets", {}).get(slug, {})
    control_comparison = {
        "editorial_vs_author_signal": target_block.get("editorial_vs_author_signal", 0.0),
    }
    strength = classify_finding_strength(
        convergence_window=best,
        hypothesis_tests=window_tests,
        control_comparison=control_comparison,
        probability_features_available=False,
    )
    strength_rows.append(
        {
            "slug": slug,
            "strength": str(strength),
            "pa": round(best.pipeline_a_score or 0.0, 3),
            "pb": round(best.pipeline_b_score or 0.0, 3),
            "window_start": best.start_date.isoformat() if best.start_date else None,
            "window_end": best.end_date.isoformat() if best.end_date else None,
            "n_families": len(best.families_converging),
            "n_sig": sum(1 for t in window_tests if t.significant),
        }
    )

strength_df = pl.DataFrame(strength_rows).sort(["strength", "pb"], descending=[False, True])
display(strength_df)

## Recommendations

1. **Re-extract embeddings for any future author whose `pb_max` regresses to 0.** The manifest regeneration that revived Pipeline B was a one-shot patch; the underlying scrape/extract path should treat embedding manifest staleness as a failure mode and re-emit on detect (rather than silently producing empty drift bundles).
2. **Keep `pipeline_b_mode = percentile` (already done).** This change is what restored cross-author comparability of `pb_max`; reverting to the prior absolute-cosine mode would re-collapse the distribution.
3. **Land G1 max_workers wiring (Phase 15 H2).** Serial 21-minute runs are workable but parallel execution unlocks the batch-of-50 author workflow that the survey gate will need.
4. **Lock pre-registration before any external publication.** `colby-hall` looks defensible but the thresholds used here (PA ≥ 0.5, PB ≥ 0.4 for AB-confirmation) were chosen *after* observing the data. Until `data/preregistration/` is locked and verified, every claim in this report is exploratory.
5. **Phase 9 (token-probability features) remains the only path to Pipeline C.** Until it lands, every "finding" is a 2-of-3 ensemble call rather than the planned 3-of-3.

## Methodology and provenance pointers

- Pre-registration scaffolding: `docs/pre_registration.md` and `data/preregistration/`.
- Feature dictionaries: `src/forensics/features/`.
- Statistical helpers (BH-FDR, effect sizes, hypothesis testing): `src/forensics/analysis/statistics.py`.
- Convergence scoring (Pipelines A/B/C, `passes_via`): `src/forensics/analysis/convergence.py`.
- Narrative + finding-strength classifier: `src/forensics/models/report.py`, `src/forensics/reporting/narrative.py`.
- Custody record: `data/analysis/corpus_custody.json`.

## Provenance

This cell prints the git SHA and the deterministic hash of `settings.analysis` so the report is auditable downstream.

In [ ]:
import subprocess
import sys
from datetime import UTC, datetime

from IPython.display import Markdown, display

from forensics.config import get_project_root, get_settings
from forensics.utils.provenance import compute_model_config_hash

settings = get_settings()
root = get_project_root()

try:
    git_sha = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=root, stderr=subprocess.STDOUT
    ).decode().strip()
except (subprocess.CalledProcessError, FileNotFoundError) as exc:
    git_sha = f"<unavailable: {exc}>"

analysis_hash = compute_model_config_hash(settings.analysis)
run_timestamp = datetime.now(UTC).isoformat()

display(
    Markdown(
        f"""| Key | Value |
|---|---|
| Git SHA | `{git_sha}` |
| Analysis-config hash | `{analysis_hash}` |
| Report rendered at | `{run_timestamp}` |
| Python | `{sys.version.split()[0]}` |
"""
    )
)